# SmartRMA hardware recognition training

MobileNetV3 Small alongside the existing local vision LLM. Four hardware classes; **no damage or return eligibility labels**. Use a Colab T4 GPU. Public sources, licenses, duplicate filtering and limits are described in training/README.md. Physical unit IDs are unavailable; the test split is exploratory.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx==1.20.1','onnxruntime==1.24.1','ImageHash==4.3.2','scikit-learn==1.6.1'],check=True)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
from pathlib import Path
Path('prepare_dataset.py').write_text('"""Download versioned public hardware photographs; audit and group before splitting.\n\nProduct recognition labels ONLY. No damage/functional/return eligibility labels.\n"""\nimport argparse\nfrom collections import Counter, defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nfrom io import BytesIO\nimport json\nfrom pathlib import Path\nimport random\nimport time\nfrom urllib.parse import urlencode\nfrom urllib.request import urlopen, Request\nimport zipfile\n\nfrom PIL import Image, ImageOps\nimport imagehash\n\nKAGGLE = \'https://www.kaggle.com/datasets/asaniczka/pc-parts-images-dataset-classification\'\nMENDELEY = \'https://data.mendeley.com/datasets/pwctygncrj/4\'\nARCHIVE_SHA = \'9404d2f13991d0803185fc29919b4709180c68817cf08060264f0b2c98b8cea6\'\nRAW_FOLDERS = {\n    \'motherboard\': \'9fa92373-7bd8-4644-bc3b-d1a4fdba2812\',\n    \'ram\': \'e866026f-21b3-4cf1-adfa-1d5948949025\',\n    \'ssd\': \'9f39e795-2989-48e3-9be7-161220e5a1e4\',\n}\n\n\ndef fetch(url):\n    for attempt in range(4):\n        try:\n            with urlopen(Request(url, headers={\'User-Agent\': \'SmartRMA-research/1.0\'}), timeout=60) as response:\n                return response.read()\n        except Exception:\n            if attempt == 3:\n                raise\n            time.sleep(1 + attempt)\n\n\ndef digest(raw):\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef prepare(root):\n    root = Path(root).resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    cache = root / \'sources\'\n    cache.mkdir(exist_ok=True)\n    archive = cache / \'pc-parts-v1.zip\'\n    if not archive.exists():\n        archive.write_bytes(fetch(\'https://www.kaggle.com/api/v1/datasets/download/\'\n                                 \'asaniczka/pc-parts-images-dataset-classification?datasetVersionNumber=1\'))\n    if digest(archive.read_bytes()) != ARCHIVE_SHA:\n        raise ValueError(\'Public archive changed: inspect the dataset version before using it.\')\n    candidates = []\n    with zipfile.ZipFile(archive) as zipped:\n        for name in sorted(zipped.namelist()):\n            bits = name.split(\'/\')\n            if len(bits) == 3 and bits[1] in {\'gpu\', \'motherboard\', \'ram\'}:\n                candidates.append((\'graphics_card\' if bits[1] == \'gpu\' else bits[1],\n                                   \'kaggle_pc_parts_v1\', name, zipped.read(name), KAGGLE,\n                                   \'ODC-By (database); underlying image rights remain with original owners\'))\n    # Include laptop RAM and motherboards too, so an SSD is not the only class\n    # with the academic dataset\'s photography background.\n    pending = []\n    for label, folder in RAW_FOLDERS.items():\n        url = \'https://data.mendeley.com/public-api/datasets/pwctygncrj/files?\' + urlencode({\n            \'folder_id\': folder, \'version\': 4, \'$start\': 0, \'$limit\': 1000})\n        entries = json.loads(fetch(url))\n        if len(entries) != 140:\n            raise ValueError(f\'Expected 140 original {label} photographs, got {len(entries)}\')\n        (cache / f\'mendeley-{label}-v4.json\').write_text(json.dumps(entries, indent=2))\n        pending.extend((label, entry) for entry in entries)\n\n    def download(item):\n        label, entry = item\n        dest = cache / (entry[\'id\'] + \'.jpg\')\n        raw = dest.read_bytes() if dest.exists() else fetch(entry[\'content_details\'][\'download_url\'])\n        if digest(raw) != entry[\'content_details\'][\'sha256_hash\']:\n            raise ValueError(\'Academic source checksum mismatch: \' + entry[\'id\'])\n        dest.write_bytes(raw)\n        return (label, \'mendeley_laptop_v4_raw\', entry[\'filename\'], raw, MENDELEY, \'CC BY 4.0\')\n\n    with ThreadPoolExecutor(max_workers=8) as pool:\n        for i, row in enumerate(pool.map(download, pending), 1):\n            candidates.append(row)\n            if i % 50 == 0:\n                print(f\'Downloaded {i}/{len(pending)} academic images\', flush=True)\n    rows, excluded, seen, conflicts = [], [], {}, set()\n    for label, source, name, raw, source_url, license_name in candidates:\n        sha = digest(raw)\n        try:\n            with Image.open(BytesIO(raw)) as image:\n                image.load()\n                image = ImageOps.exif_transpose(image).convert(\'RGB\')\n                if min(image.size) < 64:\n                    raise ValueError(\'image too small\')\n                pixel_sha = digest(str(image.size).encode() + image.tobytes())\n                phash = str(imagehash.phash(image))\n                width, height = image.size\n        except Exception as exc:\n            excluded.append({\'source\': source, \'name\': name, \'reason\': type(exc).__name__})\n            continue\n        if pixel_sha in seen:\n            if seen[pixel_sha][\'label\'] != label:\n                conflicts.add(pixel_sha)\n                excluded.append({\'source\': source, \'name\': name, \'reason\': \'conflicting_pixel_labels\',\n                                 \'also_excluded\': seen[pixel_sha][\'path\']})\n                continue\n            excluded.append({\'source\': source, \'name\': name, \'reason\': \'exact_pixel_duplicate\',\n                             \'duplicate_of\': seen[pixel_sha][\'sha256\']})\n            continue\n        path = Path(\'images\') / label / (sha + \'.jpg\')\n        dest = root / path\n        dest.parent.mkdir(parents=True, exist_ok=True)\n        dest.write_bytes(raw)\n        row = {\'path\': path.as_posix(), \'label\': label, \'condition\': \'unlabeled\',\n               \'sha256\': sha, \'pixel_sha256\': pixel_sha, \'phash\': phash,\n               \'width\': width, \'height\': height, \'source\': source,\n               \'source_filename\': name, \'source_url\': source_url, \'license\': license_name}\n        seen[pixel_sha] = row\n        rows.append(row)\n    rows = [r for r in rows if r[\'pixel_sha256\'] not in conflicts]\n    # Connected components of near-duplicates, including cross-source matches.\n    # This cannot discover all views of the same physical unit; identifiers absent.\n    parents = list(range(len(rows)))\n\n    def find(i):\n        while parents[i] != i:\n            parents[i] = parents[parents[i]]\n            i = parents[i]\n        return i\n\n    hashes = [int(row[\'phash\'], 16) for row in rows]\n    conflicting_near = set()\n    for i in range(len(rows)):\n        for j in range(i):\n            if (hashes[i] ^ hashes[j]).bit_count() <= 6:\n                if rows[i][\'label\'] != rows[j][\'label\']:\n                    conflicting_near.update([i, j])\n                parents[find(i)] = find(j)\n    bad_groups = {find(i) for i in conflicting_near}\n    groups = defaultdict(list)\n    for i, row in enumerate(rows):\n        if find(i) in bad_groups:\n            excluded.append({\'source\': row[\'source\'], \'name\': row[\'source_filename\'],\n                             \'reason\': \'conflicting_near_duplicate_group\'})\n            continue\n        group = rows[find(i)][\'sha256\']\n        row[\'group_id\'] = group\n        groups[row[\'label\'], group].append(row)\n    rows = [row for group in groups.values() for row in group]\n    # Quarantine excluded files already written during auditing, rather than\n    # counting them as usable class images or silently choosing a label.\n    usable = {row[\'path\'] for row in rows}\n    for path in (root/\'images\').glob(\'*/*.jpg\'):\n        if path.relative_to(root).as_posix() not in usable:\n            dest = root/\'quarantine\'/path.parent.name/path.name\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            path.replace(dest)\n    rng = random.Random(42)\n    for label in sorted({row[\'label\'] for row in rows}):\n        grouped = [group for (key, _), group in groups.items() if key == label]\n        rng.shuffle(grouped)\n        if len(grouped) < 10:\n            raise ValueError(f\'Not enough independent visual groups for {label}: {len(grouped)}\')\n        targets = {\'train\': .7, \'val\': .15, \'test\': .15}\n        assigned = Counter()\n        total = sum(map(len, grouped))\n        # Ensure each split has groups before filling relative target deficits.\n        for i, group in enumerate(sorted(grouped, key=len, reverse=True)):\n            split = list(targets)[i] if i < 3 else max(targets, key=lambda s: targets[s] - assigned[s]/total)\n            for row in group:\n                row[\'split\'] = split\n            assigned[split] += len(group)\n    rows.sort(key=lambda row: row[\'path\'])\n    assert len(rows) >= 501, \'Need at least 501 original images after deduplication\'\n    assert sum(row[\'split\'] == \'train\' for row in rows) >= 501, \'Need 501+ training images\'\n    (root / \'manifest.json\').write_text(json.dumps(rows, indent=2))\n    report = {\'schema_version\': 1, \'task\': \'hardware_type_classification\', \'seed\': 42,\n              \'condition_labels_available\': False, \'unique_original_images\': len(rows),\n              \'counts\': dict(Counter(row[\'label\'] for row in rows)),\n              \'split_counts\': {s: dict(Counter(r[\'label\'] for r in rows if r[\'split\'] == s)) for s in [\'train\',\'val\',\'test\']},\n              \'visual_groups\': len(groups), \'excluded\': excluded,\n              \'limitations\': [\'No damage, function, authenticity or warranty labels.\',\n                  \'Physical unit IDs absent: repeated item views can remain across splits.\',\n                  \'SSD source has limited device diversity; per-source/domain testing required.\',\n                  \'Kaggle web images can contain labels or backgrounds that act as shortcuts.\',\n                  \'Near-duplicate grouping uses pHash <= 6; it is not proof of unit independence.\']}\n    (root / \'dataset_report.json\').write_text(json.dumps(report, indent=2))\n    print(json.dumps({k: report[k] for k in [\'unique_original_images\', \'counts\', \'split_counts\', \'visual_groups\']}, indent=2), flush=True)\n    return report\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--output\', type=Path, default=Path(\'training/data/hardware\'))\n    prepare(parser.parse_args().output)\n')


In [ ]:
from pathlib import Path
Path('train_hardware.py').write_text('"""Colab/T4 transfer learning and export. Never trains damage/return labels."""\nimport argparse\nfrom collections import Counter\nimport copy\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport shutil\nimport sys\nimport time\n\nimport numpy as np\nfrom PIL import Image, ImageOps\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\nimport torchvision\nfrom torchvision import transforms as T\nfrom torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\n\nLABELS = [\'graphics_card\', \'motherboard\', \'ram\', \'ssd\']\nMEAN, STD = [.485, .456, .406], [.229, .224, .225]\n\n\nclass Photos(Dataset):\n    def __init__(self, root, rows, training=False):\n        self.root, self.rows = Path(root), rows\n        # Pad to a square before resizing: keep complete contacts/board outlines.\n        self.augment = T.Compose([T.RandomHorizontalFlip(), T.RandomRotation(15),\n                                  T.ColorJitter(.12, .12, .12, .03)]) if training else None\n        self.to_tensor = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        with Image.open(self.root / row[\'path\']) as opened:\n            image = ImageOps.exif_transpose(opened).convert(\'RGB\')\n            image = ImageOps.pad(image, (224, 224), method=Image.Resampling.BILINEAR, color=(255,255,255))\n            if self.augment:\n                image = self.augment(image)\n            return self.to_tensor(image), LABELS.index(row[\'label\'])\n\n\ndef metrics(y, predicted):\n    return {\'accuracy\': float(accuracy_score(y, predicted)),\n            \'macro_f1\': float(f1_score(y, predicted, labels=list(range(4)), average=\'macro\', zero_division=0)),\n            \'confusion_matrix\': confusion_matrix(y, predicted, labels=list(range(4))).tolist(),\n            \'classification_report\': classification_report(y, predicted, labels=list(range(4)), target_names=LABELS,\n                                                           output_dict=True, zero_division=0)}\n\n\ndef train(root, output, epochs=12):\n    root, output = Path(root), Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    random.seed(42); np.random.seed(42); torch.manual_seed(42)\n    torch.set_num_threads(2)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(42)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    print(\'Training device:\', device, torch.cuda.get_device_name(0) if device.type == \'cuda\' else \'\', flush=True)\n    rows = json.loads((root / \'manifest.json\').read_text())\n    splits = {s: [r for r in rows if r[\'split\'] == s] for s in [\'train\',\'val\',\'test\']}\n    assert len(splits[\'train\']) >= 501, \'Need 501+ original training images\'\n    for row in rows:\n        path = (root / row[\'path\']).resolve()\n        assert path.is_relative_to(root.resolve()), \'Manifest path outside data root\'\n        assert hashlib.sha256(path.read_bytes()).hexdigest() == row[\'sha256\'], \'Image changed\'\n    for key in [\'sha256\', \'pixel_sha256\', \'group_id\']:\n        sets = {s: {r[key] for r in splits[s]} for s in splits}\n        assert not (sets[\'train\'] & sets[\'val\'] or sets[\'train\'] & sets[\'test\'] or sets[\'val\'] & sets[\'test\']), key\n    for s in splits:\n        assert set(r[\'label\'] for r in splits[s]) == set(LABELS), \'Each split must contain every class\'\n    generator = torch.Generator().manual_seed(42)\n    loaders = {s: DataLoader(Photos(root, splits[s]), batch_size=32, shuffle=False, num_workers=2) for s in splits}\n    train_loader = DataLoader(Photos(root, splits[\'train\'], True), batch_size=32,\n                              shuffle=True, num_workers=2, generator=generator)\n    model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT).to(device)\n\n    # Honest transfer-learning baseline: frozen ImageNet feature nearest centroids.\n    def embeddings(loader):\n        model.eval(); vectors, labels = [], []\n        with torch.inference_mode():\n            for x, y in loader:\n                features = model.avgpool(model.features(x.to(device))).flatten(1)\n                vectors.append(nn.functional.normalize(features, dim=1).cpu())\n                labels.append(y)\n        return torch.cat(vectors), torch.cat(labels)\n\n    features, y_train = embeddings(loaders[\'train\'])\n    centroids = torch.stack([features[y_train == i].mean(0) for i in range(4)])\n    centroids = nn.functional.normalize(centroids, dim=1)\n    test_features, y_test = embeddings(loaders[\'test\'])\n    baseline_predictions = (test_features @ centroids.T).argmax(1).numpy()\n    baseline = metrics(y_test.numpy(), baseline_predictions)\n    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 4).to(device)\n    counts = Counter(r[\'label\'] for r in splits[\'train\'])\n    weights = torch.tensor([len(splits[\'train\'])/(4*counts[label]) for label in LABELS], device=device)\n    criterion = nn.CrossEntropyLoss(weight=weights)\n    optimizer = torch.optim.AdamW([\n        {\'params\': model.features.parameters(), \'lr\': 1e-4},\n        {\'params\': model.classifier.parameters(), \'lr\': 1e-3}], weight_decay=1e-4)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)\n    best, best_state, best_epoch = -1, None, None\n    history, start = [], time.monotonic()\n\n    def logits(loader):\n        model.eval(); scores, truths = [], []\n        with torch.inference_mode():\n            for x, y in loader:\n                scores.append(model(x.to(device)).cpu()); truths.append(y)\n        return torch.cat(scores), torch.cat(truths)\n\n    for epoch in range(epochs):\n        model.train(); total_loss = 0\n        for x, y in train_loader:\n            optimizer.zero_grad(set_to_none=True)\n            loss = criterion(model(x.to(device)), y.to(device))\n            loss.backward(); optimizer.step()\n            total_loss += float(loss.detach()) * len(y)\n        scheduler.step()\n        validation_logits, validation_y = logits(loaders[\'val\'])\n        score = f1_score(validation_y.numpy(), validation_logits.argmax(1).numpy(), average=\'macro\', zero_division=0)\n        history.append({\'epoch\': epoch+1, \'train_loss\': total_loss/len(splits[\'train\']), \'val_macro_f1\': float(score)})\n        print(json.dumps(history[-1]), flush=True)\n        if score > best:\n            best, best_epoch = score, epoch+1\n            best_state = copy.deepcopy(model.state_dict())\n    model.load_state_dict(best_state)\n    validation_logits, validation_y = logits(loaders[\'val\'])\n    # Fit only a scalar temperature using validation NLL, never test labels.\n    temperatures = np.linspace(.5, 4, 71)\n    nll = [float(nn.functional.cross_entropy(validation_logits/float(t), validation_y)) for t in temperatures]\n    temperature = float(temperatures[int(np.argmin(nll))])\n    test_logits, test_y = logits(loaders[\'test\'])\n    probabilities = torch.softmax(test_logits/temperature, 1).numpy()\n    predictions = probabilities.argmax(1)\n    final = metrics(test_y.numpy(), predictions)\n    report = {\n        \'task\': \'hardware_type_classification_only\', \'architecture\': \'mobilenet_v3_small\',\n        \'base_weights\': \'torchvision IMAGENET1K_V1\', \'labels\': LABELS,\n        \'dataset_counts\': {s: dict(Counter(r[\'label\'] for r in splits[s])) for s in splits},\n        \'split_method\': \'seed42 exact-pixel deduplication and pHash<=6 connected visual groups; physical IDs unknown\',\n        \'frozen_imagenet_centroid_baseline_test\': baseline, \'fine_tuned_test\': final,\n        \'test_accuracy_delta\': final[\'accuracy\']-baseline[\'accuracy\'],\n        \'best_epoch_by_validation_macro_f1\': best_epoch, \'best_validation_macro_f1\': float(best),\n        \'validation_temperature\': temperature, \'history\': history,\n        \'training_seconds\': time.monotonic()-start,\n        \'runtime\': {\'python\': sys.version, \'torch\': torch.__version__, \'torchvision\': torchvision.__version__,\n                    \'device\': str(device), \'gpu\': torch.cuda.get_device_name(0) if device.type==\'cuda\' else None},\n        \'limitations\': json.loads((root/\'dataset_report.json\').read_text())[\'limitations\'] + [\n            \'Recognition scores do not measure damage or end-to-end return decision accuracy.\',\n            \'No out-of-distribution rejection guarantee; every image maps onto these four labels.\',\n            \'No same-dataset comparison against Qwen3-VL was run.\']}\n    (output/\'metrics.json\').write_text(json.dumps(report, indent=2))\n    torch.save({\'state_dict\': model.cpu().state_dict(), \'labels\': LABELS, \'temperature\': temperature}, output/\'hardware.pt\')\n    model.eval()\n    example = torch.randn(1,3,224,224)\n    torch.onnx.export(model, example, str(output/\'hardware.onnx\'), input_names=[\'image\'], output_names=[\'logits\'],\n                      dynamic_axes={\'image\': {0: \'batch\'}, \'logits\': {0:\'batch\'}}, opset_version=17, dynamo=False)\n    import onnxruntime as ort\n    session = ort.InferenceSession(str(output/\'hardware.onnx\'), providers=[\'CPUExecutionProvider\'])\n    with torch.inference_mode():\n        torch_scores = model(example).numpy()\n    onnx_scores = session.run(None, {\'image\': example.numpy()})[0]\n    assert np.allclose(torch_scores, onnx_scores, rtol=1e-3, atol=1e-4), \'ONNX export mismatch\'\n    metadata = {\'schema_version\': 1, \'task\': \'hardware_type_classification\', \'architecture\': \'mobilenet_v3_small\',\n        \'labels\': LABELS, \'input_name\': \'image\', \'input_size\': 224, \'mean\': MEAN, \'std\': STD,\n        \'preprocessing\': \'exif_transpose_rgb_white_pad_bilinear_224\', \'temperature\': temperature,\n        \'confidence_notice\': \'Validation temperature scaled; no OOD or condition certainty guarantee.\',\n        \'model_sha256\': hashlib.sha256((output/\'hardware.onnx\').read_bytes()).hexdigest(),\n        \'metrics_sha256\': hashlib.sha256((output/\'metrics.json\').read_bytes()).hexdigest(),\n        \'dataset_manifest_sha256\': hashlib.sha256((root/\'manifest.json\').read_bytes()).hexdigest(),\n        \'advisory_only\': True, \'test_accuracy\': final[\'accuracy\'], \'test_macro_f1\': final[\'macro_f1\']}\n    (output/\'metadata.json\').write_text(json.dumps(metadata, indent=2))\n    shutil.copy2(root/\'dataset_report.json\', output/\'dataset_report.json\')\n    shutil.copy2(root/\'manifest.json\', output/\'training_manifest.json\')\n    (output/\'test_predictions.json\').write_text(json.dumps([\n        {\'path\': row[\'path\'], \'true\': row[\'label\'], \'predicted\': LABELS[int(pred)], \'probabilities\': prob.tolist()}\n        for row, pred, prob in zip(splits[\'test\'], predictions, probabilities)], indent=2))\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import ConfusionMatrixDisplay\n    fig, axis = plt.subplots(figsize=(7,6))\n    ConfusionMatrixDisplay(np.array(final[\'confusion_matrix\']), display_labels=LABELS).plot(ax=axis, colorbar=False)\n    axis.set_title(\'Held-out hardware recognition (not damage classification)\')\n    fig.tight_layout(); fig.savefig(output/\'confusion_matrix.png\', dpi=170); plt.close(fig)\n    bundle = shutil.make_archive(str(output.parent/\'smartrma_hardware_bundle\'), \'zip\', output)\n    print(\'RESULT\', json.dumps({\'baseline_test_accuracy\': baseline[\'accuracy\'], \'fine_tuned_test_accuracy\': final[\'accuracy\'],\n                              \'macro_f1\': final[\'macro_f1\'], \'bundle\': bundle}), flush=True)\n    return report\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--data\', type=Path, default=Path(\'training/data/hardware\'))\n    parser.add_argument(\'--output\', type=Path, default=Path(\'training/runs/hardware\'))\n    parser.add_argument(\'--epochs\', type=int, default=12)\n    args = parser.parse_args()\n    if not 1 <= args.epochs <= 100:\n        parser.error(\'epochs must be in 1..100\')\n    train(args.data, args.output, args.epochs)\n')


## Prepare original photographs
Downloads versioned public files. No credentials, customer images or hosted AI APIs. 971 retained images in this run; at least 501 original **training** images are enforced. Corrupt, duplicate and conflicting labels are excluded; pHash groups stay together across the split.


In [ ]:
subprocess.run([sys.executable,'-u','prepare_dataset.py','--output','data/hardware'],check=True)


## Train and export
Validation selects the epoch and temperature; the held-out test split is evaluated only after fitting. The comparison baseline uses frozen ImageNet features, not Qwen3-VL.


In [ ]:
with subprocess.Popen([sys.executable,'-u','train_hardware.py','--data','data/hardware','--output','runs/hardware','--epochs','12'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
    for line in process.stdout:
        print(line,end='',flush=True)
    assert process.wait()==0, 'Training failed; inspect output above.'


In [ ]:
import json
from pathlib import Path
r=json.loads(Path('runs/hardware/metrics.json').read_text())
print(json.dumps({k:r[k] for k in ['frozen_imagenet_centroid_baseline_test','fine_tuned_test','test_accuracy_delta']},indent=2))
from IPython.display import display, Image
display(Image('runs/hardware/confusion_matrix.png'))


## Save the artifact
Download before terminating the Colab runtime. Install using `python training/install_model.py <bundle.zip>` in SmartRMA after installing requirements-hardware.txt. The classifier is advisory only, runs locally in Node 3, and cannot bypass the T4 human requirement.


In [ ]:
from google.colab import files
files.download('runs/smartrma_hardware_bundle.zip')
